# CGAN Data Augmentation – Panoramica
Flusso completo per generare campioni sintetici delle classi rare (DISGUST, SURPRISE, NEUTRAL) con un Conditional GAN: mount Drive → load dataset → individuazione classi rare → preparazione dati → definizione modelli → training → generazione sintetici → salvataggi e check qualitativi.

## Section 1: Monta Google Drive
Necessario per leggere il dataset e salvare modelli/output su Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Section 2: Import e check GPU
Importa TensorFlow/NumPy, controlla versione e disponibilità GPU.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import h5py, os

print('TF', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## Section 3: Caricamento dataset HDF5
Carica train/val da `dataset.h5` e unisce i due split per l'analisi.

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/final_scripts/dataset/dataset.h5'
with h5py.File(DATASET_PATH, 'r') as f:
    X_train = np.array(f['X_train'])
    y_train = np.array(f['y_train'])
    X_val = np.array(f['X_val'])
    y_val = np.array(f['y_val'])
    class_names = [c.decode('utf-8') for c in f['class_names']]
X = np.concatenate([X_train, X_val])
y = np.concatenate([y_train, y_val])
print('Shapes', X.shape, y.shape)

## Section 4: Individuazione classi rare
Trova classi con frequenza <15% della classe più numerosa e visualizza la distribuzione.

In [ ]:
counts = np.bincount(y)
max_count = counts.max()
rare_idx = np.where(counts < 0.15 * max_count)[0]
rare_names = [class_names[i] for i in rare_idx]
print('Rare classes:', dict(zip(rare_idx, rare_names)))
for i, n in enumerate(class_names):
    print(f'{n:10s}: {counts[i]}')
plt.bar(class_names, counts)
plt.xticks(rotation=45)
plt.show()

## Section 5: Prepara subset classi rare
Filtra solo le classi rare, normalizza in [-1,1], rimappa etichette a indice locale e applica one-hot.

In [ ]:
mask = np.isin(y, rare_idx)
X_rare = X[mask].astype('float32') / 127.5 - 1.0
y_rare = y[mask]
label_map = {old:new for new, old in enumerate(rare_idx)}
y_remap = np.array([label_map[v] for v in y_rare])
num_classes = len(rare_idx)
y_onehot = tf.keras.utils.to_categorical(y_remap, num_classes)
print('Rare subset', X_rare.shape, y_onehot.shape)

## Section 6: Hyperparameter setup
Definisci dimensione immagine, latente, batch e LR per Adam.

In [ ]:
IMG_SHAPE = (128,128,3)
NOISE_DIM = 100
BATCH_SIZE = 32
LR = 2e-4
BETA1 = 0.5

## Section 7: Definizione modelli CGAN
Generator: noise+label → immagine. Discriminator condizionato: immagine+label → real/fake. Compila D con BCE, blocca D nel modello combinato per allenare G.

In [ ]:
def build_generator():
    noise = tf.keras.Input(shape=(NOISE_DIM,))
    label = tf.keras.Input(shape=(num_classes,))
    x = tf.keras.layers.Concatenate()([noise, label])
    x = tf.keras.layers.Dense(16*16*128, activation='relu')(x)
    x = tf.keras.layers.Reshape((16,16,128))(x)
    for f in [128,64,32]:
        x = tf.keras.layers.Conv2DTranspose(f, 4, strides=2, padding='same', activation='relu')(x)
        x = tf.keras.layers.BatchNormalization()(x)
    out = tf.keras.layers.Conv2D(3, 3, padding='same', activation='tanh')(x)
    return tf.keras.Model([noise, label], out)

def build_discriminator():
    img = tf.keras.Input(shape=IMG_SHAPE)
    label = tf.keras.Input(shape=(num_classes,))
    l = tf.keras.layers.Reshape((1,1,num_classes))(label)
    l = tf.keras.layers.UpSampling2D(size=(IMG_SHAPE[0], IMG_SHAPE[1]))(l)
    x = tf.keras.layers.Concatenate()([img, l])
    for f in [64,128,256]:
        x = tf.keras.layers.Conv2D(f, 4, strides=2, padding='same')(x)
        x = tf.keras.layers.LeakyReLU(0.2)(x)
        x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Flatten()(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    return tf.keras.Model([img, label], out)

G = build_generator()
D = build_discriminator()
D.compile(optimizer=tf.keras.optimizers.Adam(LR, beta_1=BETA1), loss='binary_crossentropy', metrics=['accuracy'])
noise_in = tf.keras.Input(shape=(NOISE_DIM,))
label_in = tf.keras.Input(shape=(num_classes,))
fake_img = G([noise_in, label_in])
D.trainable = False
valid = D([fake_img, label_in])
GAN = tf.keras.Model([noise_in, label_in], valid)
GAN.compile(optimizer=tf.keras.optimizers.Adam(LR, beta_1=BETA1), loss='binary_crossentropy')
G.summary(); D.summary()

## Section 8: Training loop (1:1 D/G)
Allena discriminator e generator con rapporto 1:1 sui batch delle classi rare; logga le loss.

In [ ]:
def train(epochs=200):
    d_hist, g_hist = [], []
    ds = tf.data.Dataset.from_tensor_slices((X_rare, y_onehot)).shuffle(len(X_rare)).batch(BATCH_SIZE)
    for epoch in range(epochs):
        d_loss_epoch = g_loss_epoch = 0.0
        steps = 0
        for real_imgs, labels in ds:
            bs = real_imgs.shape[0]
            noise = tf.random.normal((bs, NOISE_DIM))
            fake_imgs = G([noise, labels], training=True)
            real_y = tf.ones((bs,1))
            fake_y = tf.zeros((bs,1))
            d_loss_real = D.train_on_batch([real_imgs, labels], real_y)[0]
            d_loss_fake = D.train_on_batch([fake_imgs, labels], fake_y)[0]
            d_loss = 0.5*(d_loss_real + d_loss_fake)
            noise = tf.random.normal((bs, NOISE_DIM))
            g_loss = GAN.train_on_batch([noise, labels], real_y)
            d_loss_epoch += d_loss; g_loss_epoch += g_loss; steps += 1
        d_hist.append(d_loss_epoch/steps); g_hist.append(g_loss_epoch/steps)
        if (epoch+1) % 20 == 0:
            print(f'Epoch {epoch+1}/{epochs} - D: {d_hist[-1]:.3f} - G: {g_hist[-1]:.3f}')
    return d_hist, g_hist

d_losses, g_losses = train(epochs=120)
plt.plot(d_losses, label='D'); plt.plot(g_losses, label='G'); plt.legend(); plt.show()

## Section 9: Generazione sintetici
Per ogni classe rara genera campioni (≈25% del max), denormalizza a uint8 per salvataggio.

In [ ]:
target_per_class = int(max_count * 0.25)
synth_images = []
synth_labels = []
for cid in range(num_classes):
    noise = tf.random.normal((target_per_class, NOISE_DIM))
    labels = np.zeros((target_per_class, num_classes), dtype=np.float32)
    labels[:, cid] = 1
    imgs = G.predict([noise, labels], verbose=0)
    synth_images.append(imgs)
    synth_labels += [cid]*target_per_class
synth_images = np.concatenate(synth_images)
synth_images_uint8 = ((synth_images + 1)/2 * 255).astype(np.uint8)
synth_labels = np.array(synth_labels)
print('Synthetic set', synth_images_uint8.shape, synth_labels.shape)

## Section 10: Anteprima sintetici
Visualizza una griglia di esempi generati per controllo qualitativo.

In [ ]:
fig, ax = plt.subplots(2,5, figsize=(10,4))
for i in range(10):
    r, c = divmod(i,5)
    ax[r,c].imshow(synth_images_uint8[i])
    ax[r,c].axis('off')
plt.show()

## Section 11: Salvataggio modelli e dataset sintetico
Salva generator/discriminator e i file `.npy` (immagini, etichette, indici classi rare) su Drive.

In [ ]:
OUT_DIR = '/content/drive/MyDrive/GAN_augmentation'
os.makedirs(OUT_DIR, exist_ok=True)
G.save(os.path.join(OUT_DIR, 'generator_cgan.h5'))
D.save(os.path.join(OUT_DIR, 'discriminator_cgan.h5'))
np.save(os.path.join(OUT_DIR, 'synthetic_images.npy'), synth_images_uint8)
np.save(os.path.join(OUT_DIR, 'synthetic_labels.npy'), synth_labels)
np.save(os.path.join(OUT_DIR, 'rare_class_indices.npy'), rare_idx)
print('Saved to', OUT_DIR)